# Hypothesis 2 Validation:
## Small Data Modelling & Evaluating
---

## Hypothesis
Increasing dataset size and diversity improves accuracy.

## Objectives
- Train the same CNN model from the full dataset, but with a smaller subset (10% of the dataset)
- Compare performance metrics to validate Hypothesis 2.

## Inputs
* inputs/datasets/animals/small-data/train/
* inputs/datasets/animals/image/validation/
* inputs/datasets/animals/image/test/
* Image shape embeddings: outputs/v1/image_shape.pkl

## Outputs
- Image distribution plots for small data training, validation, and testing.
- Image augmentation pipeline.
- ML learning curves for both runs.
- Comparison table (subset vs full dataset).
- Model evaluation stored in pickle file.
- Predictions on random test images.
  
---

### Import libraries
A full import list for Notebook 04

In [11]:
# Core Python & utilities
import os
import random
import joblib
import numpy as np
import pandas as pd

# Plotting & Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.image import imread

# Machine Learning / Deep Learning
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Metrics
from sklearn.metrics import classification_report, confusion_matrix

# Style for plots
sns.set_style("white")


## Set working directory

In [12]:
cwd = os.getcwd()
os.chdir('/workspaces/Animal_detection_camera')
print("You set a new current directory")

work_dir = os.getcwd()
work_dir

You set a new current directory


'/workspaces/Animal_detection_camera'

## Set input directories
Set train, validation and test paths

In [13]:
my_data_dir = 'inputs/datasets/animals'
train_path = my_data_dir + '/small-data/train'
val_path = my_data_dir + '/image/validation'
test_path = my_data_dir + '/image/test'

## Set output directory

In [14]:
version = 'v1'
file_path = f'outputs/{version}/small-data'
os.makedirs(file_path, exist_ok=True)

print(f"Saving results to: {file_path}")

Saving results to: outputs/v1/small-data


## Set labels

In [ ]:
labels = os.listdir(train_path)

print(
    f"Project Labels: {labels}"
)

## Set Image Shape

In [ ]:
image_shape = joblib.load(filename=f"outputs/{version}/image_shape.pkl")
image_shape = (128, 128, 3)
print("Using image shape:", image_shape)

---

# Number of images in train, test and validation data
---

## Create subset


In [ ]:
def create_subset(original_dir, subset_dir, fraction=0.1):
    os.makedirs(subset_dir, exist_ok=True)
    for class_name in os.listdir(original_dir):
        class_path = os.path.join(original_dir, class_name)
        if not os.path.isdir(class_path):
            continue
        subset_class_path = os.path.join(subset_dir, class_name)
        os.makedirs(subset_class_path, exist_ok=True)

        files = os.listdir(class_path)
        sample_size = max(1, int(len(files) * fraction))
        sampled_files = random.sample(files, sample_size)

        for f in sampled_files:
            shutil.copy(os.path.join(class_path, f), os.path.join(subset_class_path, f))

create_subset("inputs/datasets/animals/image/train",
    "inputs/datasets/animals/small-data",
    fraction=0.1)


## Data Generators 
Small vs Full Datasets

In [ ]:
datagen = ImageDataGenerator(rescale=1./255)

train_small_gen = datagen.flow_from_directory(
    "inputs/datasets/animals/small-data", target_size=(128,128), batch_size=32, class_mode='categorical')

train_full_gen = datagen.flow_from_directory(
    "inputs/datasets/animals/image/train", target_size=(128,128), batch_size=32, class_mode='categorical')

val_gen = datagen.flow_from_directory(
    "inputs/datasets/animals/image/validation", target_size=(128,128), batch_size=32, class_mode='categorical')


### Plot frequency distributio across all species.